In [1]:
# --- Step 1: Import All Necessary Libraries ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score

print("Libraries imported successfully.")

# --- Step 2: Load and Clean the Data ---
try:
    df = pd.read_csv('House_Price_Prediction.csv')
    # Remove rows with a price of 0, which are invalid
    df = df[df['price'] != 0]
    print(f"Data loaded successfully. Shape after cleaning: {df.shape}")
except FileNotFoundError:
    print("Error: House_Price_Prediction.csv not found. Please ensure the file is in the same directory.")

# --- Step 3: Feature Engineering ---
# Create 'house_age' and 'is_renovated' for better model intuition
df['house_age'] = 2014 - df['yr_built']
df['is_renovated'] = np.where(df['yr_renovated'] == 0, 0, 1)
print("Feature engineering complete: 'house_age' and 'is_renovated' created.")

# --- Step 4: Define Features (X) and Target (y) ---
# This is our final, stable feature set that excludes the problematic 'sqft_lot'
target = 'price'
numerical_features = ['bedrooms', 'bathrooms', 'sqft_living', 'floors', 'waterfront', 
                      'view', 'condition', 'sqft_above', 'sqft_basement', 
                      'house_age', 'is_renovated']
categorical_features = ['city']
features = numerical_features + categorical_features

X = df[features]
y = df[target]

# Apply the log transformation to the target variable for model stability
y_log = np.log1p(y)
print("Features and target defined. Log transformation applied to target.")

# --- Step 5: Split Data into Training and Testing Sets ---
X_train, X_test, y_train_log, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)
# We also need the original, non-transformed prices for our final evaluation
original_y_test = y.loc[y_test.index]
print(f"Data split into training ({X_train.shape[0]} rows) and testing ({X_test.shape[0]} rows) sets.")

# --- Step 6: Create a Robust Preprocessing Pipeline ---
# This is critical for Deep Learning:
# - StandardScaler makes sure all numerical features have a similar scale.
# - OneHotEncoder converts the 'city' column into a numerical format.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])
print("Preprocessing pipeline created.")

# --- Step 7: Define, Train, and Evaluate the Deep Learning Model ---
# We bundle the preprocessor and the MLPRegressor into a single pipeline
mlp_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('regressor', MLPRegressor(hidden_layer_sizes=(100, 50),
                                                          activation='relu',
                                                          solver='adam',
                                                          max_iter=500,
                                                          random_state=42))])

print("\nTraining the Deep Learning (MLP) model... (This may take a moment)")
mlp_pipeline.fit(X_train, y_train_log)
print("Model training complete!")

# Make predictions (these will be on the log scale)
log_predictions = mlp_pipeline.predict(X_test)

# Convert the predictions back to the original dollar scale for interpretation
final_predictions = np.expm1(log_predictions)

# Calculate the final performance metrics
rmse = np.sqrt(mean_squared_error(original_y_test, final_predictions))
r2 = r2_score(original_y_test, final_predictions)

# --- Step 8: Display the Final Results ---
print("\n--- Final Deep Learning (MLP) Model Results ---")
print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
print(f"R-squared (R2 Score): {r2:.4f}")

Libraries imported successfully.
Data loaded successfully. Shape after cleaning: (4551, 18)
Feature engineering complete: 'house_age' and 'is_renovated' created.
Features and target defined. Log transformation applied to target.
Data split into training (3640 rows) and testing (911 rows) sets.
Preprocessing pipeline created.

Training the Deep Learning (MLP) model... (This may take a moment)
Model training complete!

--- Final Deep Learning (MLP) Model Results ---
Root Mean Squared Error (RMSE): $16,480,917.29
R-squared (R2 Score): -1824.7708


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score

# --- Step 1: Load and Clean Data ---
df = pd.read_csv('House_Price_Prediction.csv')
df = df[df['price'] != 0]

# --- Step 2: Define Features (X) and Target (y) using ORIGINAL columns ---
target = 'price'
# We are using the original numerical features, including sqft_lot, yr_built, etc.
numerical_features = ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 
                      'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement', 
                      'yr_built', 'yr_renovated']
categorical_features = ['city']
features = numerical_features + categorical_features

X = df[features]
y = df[target]

# We will still use the log-transformed target, as it's crucial for model stability
y_log = np.log1p(y)

# --- Step 3: Train-Test Split ---
X_train, X_test, y_train_log, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)
original_y_test = y.loc[y_test.index]

# --- Step 4: Create the Preprocessing Pipeline ---
# This pipeline scales the numerical features and one-hot encodes the city
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# --- Step 5: Define All Models ---
lr_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', LinearRegression())])
rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))])
mlp_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42))])

models = {
    "Linear Regression": lr_pipeline,
    "Random Forest": rf_pipeline,
    "Deep Learning (MLP)": mlp_pipeline
}

results = {}

# --- Step 6: Train and Evaluate Each Model ---
print("--- Training Models on Original Feature Set ---")
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train_log)
    
    log_predictions = model.predict(X_test)
    final_predictions = np.expm1(log_predictions)
    
    rmse = np.sqrt(mean_squared_error(original_y_test, final_predictions))
    r2 = r2_score(original_y_test, final_predictions)
    results[name] = {'RMSE': rmse, 'R2': r2}
    print(f"{name} training complete.")

# --- Step 7: Print Final Comparison Table ---
print("\n--- Final Comparison on Original Features ---")
for name, metrics in results.items():
    print(f"{name:<22} RMSE: ${metrics['RMSE']:,.2f} | R2: {metrics['R2']:.4f}")

--- Training Models on Original Feature Set ---
Training Linear Regression...
Linear Regression training complete.
Training Random Forest...
Random Forest training complete.
Training Deep Learning (MLP)...
Deep Learning (MLP) training complete.

--- Final Comparison on Original Features ---
Linear Regression      RMSE: $1,051,829.96 | R2: -6.4366
Random Forest          RMSE: $235,640.28 | R2: 0.6268
Deep Learning (MLP)    RMSE: $1,849,830.90 | R2: -22.0010


In [3]:
# Create the 'is_renovated' feature if it doesn't exist
if 'is_renovated' not in df.columns:
    df['is_renovated'] = np.where(df['yr_renovated'] == 0, 0, 1)

# Group by the new feature and calculate key metrics
renovation_analysis = df.groupby('is_renovated').agg(
    average_price=('price', 'mean'),
    average_sqft_living=('sqft_living', 'mean'),
    number_of_houses=('price', 'count')
).reset_index()

# Make the labels more readable for the table
renovation_analysis['is_renovated'] = renovation_analysis['is_renovated'].map({0: 'No', 1: 'Yes'})

print(renovation_analysis)

  is_renovated  average_price  average_sqft_living  number_of_houses
0           No  571333.256274          2227.937916              2706
1          Yes  538212.442004          1992.209214              1845
